In [0]:
# Imports and configuration
import json
import time
import requests
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
import mlflow

In [0]:
# Source and target tables
SILVER_PRODUCTS  = "main.silver_techmart.products"
LLM_EXTRACTED    = "main.silver_techmart.llm_extracted"


In [0]:
# Groq API configuration (replacing Gemini due to free tier quota limitations)
GROQ_MODEL   = "llama-3.1-8b-instant"
GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"
GROQ_API_KEY = dbutils.secrets.get(scope="techmart", key="groq-api-key")

print(f"✅ Groq API Key loaded: [REDACTED]")
print(f"Model: {GROQ_MODEL}")

In [0]:
# Load prompt configuration from versioned config file
PROMPTS_PATH     = "/Workspace/Repos/ts.maira@gmail.com/techmart-catalog-pipeline/config/prompts.json"

with open(PROMPTS_PATH, "r") as f:
    prompts_config = json.load(f)

PROMPT_VERSION        = prompts_config["extraction_prompt"]["version"]
PROMPT_TEMPLATE       = prompts_config["extraction_prompt"]["template"]
ALLOWED_SUBCATEGORIES = prompts_config["extraction_prompt"]["allowed_subcategories"]

print(f"✅ Config loaded")
print(f"Prompt version      : {PROMPT_VERSION}")
print(f"Allowed subcategories: {ALLOWED_SUBCATEGORIES}")

In [0]:
# Read Silver products table
df_products = spark.table(SILVER_PRODUCTS)
print(f"Products to process: {df_products.count()}")
display(df_products.select("product_id", "product_description"))

In [0]:
# LLM call function using Groq API
# Groq uses the OpenAI-compatible API format making it easy to swap providers
# We renamed from call_gemini to call_llm to be provider-agnostic

def call_llm(product_description: str, api_key: str) -> dict:
    """
    Calls the Groq API to extract structured product information.
    Returns a dict with name, brand, sub_category.
    Returns None if the call fails or response cannot be parsed.
    """
    # Build the prompt from the versioned template
    prompt = PROMPT_TEMPLATE.format(
        allowed_subcategories=", ".join(ALLOWED_SUBCATEGORIES),
        product_description=product_description
    )

    # Build request payload in OpenAI-compatible format
    payload = {
        "model": GROQ_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1,
        "max_tokens": 200
    }

    # Make the API call
    response = requests.post(
        GROQ_API_URL,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_key}"
        },
        json=payload,
        timeout=30
    )

    if response.status_code != 200:
        return None

    # Extract the text from the response
    response_text = response.json()["choices"][0]["message"]["content"]

    # Clean markdown formatting if model adds it
    response_text = response_text.strip().replace("```json", "").replace("```", "").strip()

    # Parse and return JSON
    return json.loads(response_text)

In [0]:
# Process all products with MLflow tracing
# MLflow logs every LLM call: prompt, response, latency, success/failure
# This gives us full traceability and auditability of all LLM interactions

# Creates a folder in MLflow called techmart-llm-extraction where all our logs will be saved.
mlflow.set_experiment("/Users/ts.maira@gmail.com/techmart-llm-extraction")

# Collect products as a list to iterate row by row
products = df_products.select("product_id", "product_description").collect()

results = []

with mlflow.start_run(run_name=f"extraction_{PROMPT_VERSION}"):
    
    # Log prompt metadata for traceability
    mlflow.log_param("prompt_version", PROMPT_VERSION)
    mlflow.log_param("model", GROQ_MODEL)
    mlflow.log_param("provider", "groq")
    mlflow.log_param("total_products", len(products))
    
    success_count = 0
    failure_count = 0
    
    for row in products:
        product_id          = row["product_id"]
        product_description = row["product_description"]
        
        start_time = time.time()
        
        try:
            # Call Gemini API
            extracted = call_llm(product_description, GROQ_API_KEY)
            latency   = round(time.time() - start_time, 3)
            
            if extracted and all(k in extracted for k in ["name", "brand", "sub_category"]):
                results.append({
                    "product_id"         : str(product_id),
                    "product_description": product_description,
                    "extracted_name"     : extracted.get("name"),
                    "extracted_brand"    : extracted.get("brand"),
                    "extracted_sub_category": extracted.get("sub_category"),
                    "llm_status"         : "success",
                    "llm_latency_seconds": str(latency),
                    "prompt_version"     : PROMPT_VERSION
                })
                success_count += 1
            else:
                raise ValueError("Missing keys in response")
                
        except Exception as e:
            latency = round(time.time() - start_time, 3)
            results.append({
                "product_id"            : str(product_id),
                "product_description"   : product_description,
                "extracted_name"        : None,
                "extracted_brand"       : None,
                "extracted_sub_category": None,
                "llm_status"            : "failed",
                "llm_latency_seconds"   : str(latency),
                "prompt_version"        : PROMPT_VERSION
            })
            failure_count += 1
        
        # Small delay to respect API rate limits
        time.sleep(0.5)
    
    # Log summary metrics to MLflow
    mlflow.log_metric("success_count", success_count)
    mlflow.log_metric("failure_count", failure_count)
    mlflow.log_metric("success_rate", round(success_count / len(products), 3))
    
    print(f"✅ Processing complete")
    print(f"Success : {success_count}")
    print(f"Failed  : {failure_count}")

In [0]:
# Convert results to Spark DataFrame and save as Delta table

df_extracted = spark.createDataFrame(results)

(df_extracted.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(LLM_EXTRACTED))

print(f"✅ Written: {LLM_EXTRACTED}")
display(df_extracted)

In [0]:
# Validate results
print("=== VALIDATION ===")
print(f"Total rows    : {df_extracted.count()}")
print(f"Success rows  : {df_extracted.filter(F.col('llm_status') == 'success').count()}")
print(f"Failed rows   : {df_extracted.filter(F.col('llm_status') == 'failed').count()}")

display(df_extracted.select(
    "product_id",
    "product_description",
    "extracted_name",
    "extracted_brand",
    "extracted_sub_category",
    "llm_status",
    "llm_latency_seconds"
))